# ASR Project Demo Notebook

This notebook demonstrates how to use the Automatic Speech Recognition (ASR) system:
- Clone the repository and install dependencies
- Download model checkpoints
- Run inference on a sample dataset
- Calculate WER and CER metrics
- Run inference on your own custom dataset

## 1. Setup: Clone Repository and Install Dependencies

First, we'll clone the repository and install all required packages.

In [ ]:
# Clone the repository
!git clone https://github.com/setday/hse-dla-hw1-2025.git asr_project 2>&1 | grep -v 'already exists' || echo 'Repository ready'
%cd asr_project

In [ ]:
# Install all required dependencies
!pip install -q -r requirements.txt || echo 'Dependencies installed'

## 2. Download Model Checkpoints

Download the pretrained model checkpoint and any required resources. Update the URL to point to your actual checkpoint location.

In [ ]:
from pathlib import Path

checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_file = checkpoint_dir / "model_best.pth"

# Replace this URL with your actual Google Drive or other hosting URL
# Example: https://drive.google.com/uc?id=YOUR_FILE_ID
checkpoint_url = "https://drive.google.com/uc?id=1A24QSbFNdYYvNn921B21r8dOMWFQRuRS"

!gdown {checkpoint_url} -O {checkpoint_file} || echo "Checkpoint already downloaded"

## 3. Run Inference on Sample Dataset

This section demonstrates how to run inference using the `inference.py` script on a provided dataset.

In [ ]:
# Run inference on a sample dataset
# Using the LibriSpeech test-clean dataset (default in stable_eval config)
# This may take a few minutes depending on dataset size and GPU availability.

inference_output_dir = Path("saved/demo_predictions")

!python inference.py -cn=inference \
    inferencer.from_pretrained={checkpoint_file} \
    inferencer.save_path=demo_predictions

## 4. Calculate Metrics (WER and CER)

Use the `calc_metrics.py` script to calculate Word Error Rate (WER) and Character Error Rate (CER).

In [ ]:
# Extract predictions from inference output
# The inference script saves files with format: output_{utterance_id}.txt
# We need to extract just the predicted text for metric calculation

from pathlib import Path

def extract_predictions_from_inference_output(inference_dir):
    """
    Extract predictions from inference output files.
    Inference output has format:
        Target: <ground_truth_text>
        Predicted: <predicted_text>
    """
    predictions_dir = Path(inference_dir) / "predictions"
    predictions_dir.mkdir(parents=True, exist_ok=True)
    
    inference_path = Path(inference_dir) / "test"  # "test" is the default partition
    
    output_files = list(inference_path.glob("output_*.txt"))
    print(f"Found {len(output_files)} inference output files\n")
    
    for output_file in output_files:
        with open(output_file, "r", encoding="utf-8") as f:
            content = f.read()
        
        # Extract predicted text
        lines = content.split("\n")
        predicted_text = ""
        for line in lines:
            if line.startswith("Predicted:"):
                predicted_text = line.replace("Predicted:", "").strip()
                break
        
        # Save to new file with same ID
        pred_file = predictions_dir / output_file.name.replace("output_", "")
        with open(pred_file, "w", encoding="utf-8") as f:
            f.write(predicted_text)
    
    return predictions_dir

predictions_dir = extract_predictions_from_inference_output(inference_output_dir)
print(f"+ Predictions extracted to {predictions_dir}")

In [ ]:
# Run metrics calculation
!python calc_metrics.py \
    --ground_truth_dir {str(inference_output_dir)} \
    --predicted_dir {str(predictions_dir)} \
    --verbose

## 5. Run Inference on Your Own Custom Dataset

This section shows how to run inference on your own dataset using the custom directory format.

### 5.1 Prepare Your Dataset

Your dataset should be organized as follows:

```
my_dataset/
|-- audio/
|   |-- sample_001.wav (or .mp3, .flac, .m4a)
|   |-- sample_002.wav
|   |-- ...
|-- transcriptions/ (optional - only needed for evaluation)
    |-- sample_001.txt
    |-- sample_002.txt
    |-- ...
```

Each transcription file should contain a single line with the ground truth text.

### 5.2 Dataset on Google Drive

If your dataset is stored on Google Drive:
1. Share the folder with view permissions
2. Get the folder ID from the URL: `https://drive.google.com/drive/folders/{FOLDER_ID}`
3. Mount Google Drive in the cell below and set the path

In [ ]:
# Mount Google Drive (if running in Colab)
import sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted")
else:
    print("Not running in Colab. Skip this cell if running locally.")

In [ ]:
# Configuration for custom dataset inference

# TODO: Set these paths to your dataset
CUSTOM_AUDIO_DIR = "/path/to/your/audio/directory"  # UPDATE THIS
CUSTOM_TRANSCRIPTION_DIR = "/path/to/your/transcriptions/directory"  # UPDATE THIS (optional)

# Example paths (uncomment if using Google Drive):
# CUSTOM_AUDIO_DIR = "/content/drive/My Drive/my_dataset/audio"
# CUSTOM_TRANSCRIPTION_DIR = "/content/drive/My Drive/my_dataset/transcriptions"

# Or local path example:
# CUSTOM_AUDIO_DIR = "./data/my_custom_dataset/audio"
# CUSTOM_TRANSCRIPTION_DIR = "./data/my_custom_dataset/transcriptions"

In [ ]:
# Run inference on custom dataset
!python inference.py -cn=inference \
    datasets=custom_dir \
    custom_audio_dir={CUSTOM_AUDIO_DIR} \
    custom_transcription_dir={CUSTOM_TRANSCRIPTION_DIR} \
    inferencer.from_pretrained={checkpoint_file} \
    inferencer.save_path=custom_predictions

In [ ]:
# Calculate metrics for custom dataset (if ground truth is available)

# Extract predictions first
predictions_custom_dir = extract_predictions_from_inference_output("saved/custom_predictions")

!python calc_metrics.py \
    --ground_truth_dir {str(gt_dir)} \
    --predicted_dir {str(predictions_custom_dir)} \
    --verbose